# Calculate the spatial gradient of a field on the mesh.

In [3]:
# Test evaluate ufl expression

from FELiCS.SpaceDisc.FELiCSMesh import FELiCSMesh

meshFileName = "./square_mesh.msh"
coordinateSystem = "Cartesian"

# Create a FELiCS mesh from the gmsh file
mesh = FELiCSMesh(coordinateSystem=coordinateSystem, meshFileName=meshFileName)

from FELiCS.SpaceDisc.FEMSpaces import getFELiCSSpace

space = getFELiCSSpace(mesh, order=2, dim=1)

from FELiCS.Fields.Field import Field

Field1_init = Field(space, mesh=mesh)
Field1_init.importH5("function_values_sine")
Field1 = Field(space, mesh=mesh)
Field1.importH5("function_values_sine")

from ufl import TestFunction, dx, conj
import dolfinx.fem
v = TestFunction(space)
expression = Field1.function * conj(v) * dx

Field1.evaluateUflExpression(expression)    

result = Field1.getCoefficientArray()-Field1_init.getCoefficientArray()
print(sum(result))



Info     | FELiCSMesh.py          | __init__                   (line 28  ) : Opening mesh file: ./square_mesh.msh
Info     | FELiCSMesh.py          | __init__                   (line 36  ) : Mesh contains 513 nodes and 1024 elements


(-2.4490981555554987e-15+0j)


In [23]:
# Second test: Evaluate a gradient

from FELiCS.SpaceDisc.FELiCSMesh import FELiCSMesh

meshFileName = "./square_mesh.msh"
coordinateSystem = "Cartesian"

# Create a FELiCS mesh from the gmsh file
mesh = FELiCSMesh(coordinateSystem=coordinateSystem, meshFileName=meshFileName)

from FELiCS.SpaceDisc.FEMSpaces import getFELiCSSpace

space_scalar = getFELiCSSpace(mesh, order=2, dim=1)
space_vec = getFELiCSSpace(mesh, order=2, dim=2)

from FELiCS.Fields.Field import Field

Field1 = Field(space_scalar, mesh=mesh)
Field1.importH5("function_values_sine")
Field_grad = Field(space_vec, mesh=mesh)
Field_grad_sol = Field(space_vec, mesh=mesh)
Field_grad_sol.importH5("grad_solution")

from ufl import TestFunction, dx, conj
from FELiCS.Misc.tensorUtils import iGrad, iConj, iDot, Tensor
import dolfinx.fem
J_hat = mesh.coordinateSystem.J_hat
v = TestFunction(space_vec)
v_tens = Tensor(v, CoordSys=mesh.coordinateSystem, hasSpectralDimension=False)
expression = iDot(iGrad(Field1.getTensor()), iConj(v_tens)).ufl_tens * J_hat* dx

Field_grad.evaluateUflExpression(expression)    

print(sum(Field_grad.getCoefficientArray() - Field_grad_sol.getCoefficientArray()))

Info     | FELiCSMesh.py          | __init__                   (line 28  ) : Opening mesh file: ./square_mesh.msh
Info     | FELiCSMesh.py          | __init__                   (line 36  ) : Mesh contains 513 nodes and 1024 elements


(-0.14875251731100053+0j)


In [22]:
import numpy as np

def test_function_2(x):
    """Sine wave function: f(x,y) = sin(2*pi*x) * cos(2*pi*y)"""
    return np.sin(2 * np.pi * x[0]) * np.cos(2 * np.pi * x[1])

def dtest_dx(x):
    """Derivative of the sine wave function with respect to x: df/dx = 2*pi*cos(2*pi*x)*cos(2*pi*y)"""
    return 2 * np.pi * np.cos(2 * np.pi * x[0]) * np.cos(2 * np.pi * x[1])

def dtest_dy(x):
    """Derivative of the sine wave function with respect to y: df/dy = -2*pi*sin(2*pi*x)*sin(2*pi*y)"""
    return -2 * np.pi * np.sin(2 * np.pi * x[0]) * np.sin(2 * np.pi * x[1])

# Get coordinates of all DOFs in the P2 space
dof_coordinates = space.tabulate_dof_coordinates()
print(f"Number of DOFs: {len(dof_coordinates)}")
print(f"DOF coordinates shape: {dof_coordinates.shape}")

# Calculate function values at DOF coordinates
function_values_1 = np.zeros(len(dof_coordinates))
function_values_2 = np.zeros(len(dof_coordinates))

for i, coord in enumerate(dof_coordinates):
    function_values_1[i] = dtest_dx(coord)
    function_values_2[i] = dtest_dy(coord)

values = np.hstack((function_values_1, function_values_2))
np.save("grad_solution.npy", values)


Number of DOFs: 1969
DOF coordinates shape: (1969, 3)
